
# Example: Variational Autoencoder


$a^2$

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from numpyro.distributions.flows import InverseAutoregressiveTransform
from numpyro.nn import AutoregressiveNN

import numpy as np
import argparse
import inspect
import os
import time

import matplotlib.pyplot as plt

from jax import jit, lax, random
from jax.example_libraries import stax
import jax.numpy as jnp
from jax.random import PRNGKey
import jax

import numpyro
from numpyro import optim
import numpyro.distributions as dist
from numpyro.examples.datasets import MNIST, load_dataset
from numpyro.infer import SVI, Trace_ELBO

from types import SimpleNamespace

from numpyro.optim import Adagrad

from numpyro.contrib.einstein import RBFKernel
from CustomModules.mixture_guide_impl_source import MixtureGuidePredictive
from CustomModules.stein_impl_source import SteinVI

from CustomModules.normalizing_flow import normalizing_flow

import vae_example

from functools import partial


/home/kenn50/miniconda3/envs/wsl-test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Temp dir in kernel
RESULTS_DIR = os.path.abspath(
    os.path.join(os.path.dirname(inspect.getfile(lambda: None)), ".results")
)
RESULTS_DIR = "./smi_augmented_results"
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:


def encoder(hidden_dim, z_dim):
    return stax.serial(
        stax.Dense(hidden_dim, W_init=stax.randn()),
        stax.Softplus,
        stax.FanOut(2),
        stax.parallel(
            stax.Dense(z_dim, W_init=stax.randn()),
            stax.serial(stax.Dense(z_dim, W_init=stax.randn()), stax.Softplus),
        ),
    )


In [4]:

def decoder(hidden_dim, out_dim):
    return stax.serial(
        stax.Dense(hidden_dim, W_init=stax.randn()),
        stax.Softplus,
        stax.Dense(out_dim, W_init=stax.randn()),
        stax.Sigmoid,
    )



In [5]:
def model(batch, hidden_dim=400, z_dim=50):
    batch = jnp.reshape(batch, (batch.shape[0], -1))
    batch_dim, out_dim = jnp.shape(batch)

    # m is the global shared variable
    m = numpyro.sample("m", dist.Normal(0, 1).expand([z_dim]).to_event())


    # Configue Flow
    flow_transform = numpyro.module(
        "flow", 
        normalizing_flow(
            input_dim=z_dim, 
            hidden_dims=[z_dim, z_dim],
            steps=3,
            inv = False
        ),
        input_shape=(batch_dim, z_dim)
    )()

    base_dist = dist.Normal(m, jnp.ones(z_dim))
    flow_dist = dist.TransformedDistribution(base_dist, flow_transform)


    #Configure decoder
    decode = numpyro.module("decoder", decoder(hidden_dim, out_dim), (batch_dim, z_dim))
    with numpyro.plate("batch", 60000, subsample_size=batch_dim):
        
        z = numpyro.sample("z", flow_dist)
        img_loc = decode(z)
        return numpyro.sample("obs", dist.Bernoulli(img_loc).to_event(1), obs=batch)

In [6]:
def encoder(hidden_dim, z_dim):
    return stax.serial(
        stax.Dense(hidden_dim, W_init=stax.randn()),
        stax.Softplus,
        stax.FanOut(2),
        stax.parallel(
            stax.Dense(z_dim, W_init=stax.randn()),
            stax.serial(stax.Dense(z_dim, W_init=stax.randn()), stax.Exp),
        ),
    )


def guide(batch, hidden_dim=400,  z_dim = 50):
    batch = jnp.reshape(batch, (batch.shape[0], -1))
    batch_dim, out_dim = jnp.shape(batch)
    
    a = numpyro.param("a", jnp.zeros(z_dim))
    B = numpyro.param("B", jnp.ones(z_dim), constraint=dist.constraints.positive)

    m = numpyro.sample("m", dist.Normal(a, B).to_event())

    encode = numpyro.module("encoder", encoder(hidden_dim, z_dim), (batch_dim, out_dim + z_dim))


    # Concatanate it for the encoder
    tiled = jnp.tile(m, (batch_dim, 1))
    concat_input = jnp.concat([batch, tiled], axis=1)


    z_loc, z_std = encode(concat_input)
    with numpyro.plate("batch", 60000, subsample_size=batch_dim):
        d = dist.Normal(z_loc, z_std).to_event(1)
        
        z = numpyro.sample("z", d)
        return z



In [7]:
def time_f(func, active = False):
    def time_func(*args, **kwargs):
        t = time.time()
        result = func(*args, **kwargs)
        if(active):
            print(f"took {time.time() - t}s to run {func.__name__}")
        return result
    return time_func


def binarize(rng_key, batch):
    return random.bernoulli(rng_key, batch).astype(batch.dtype)


args = SimpleNamespace()
args.num_epochs = 15
args.learning_rate =5.0e-3
args.batch_size = 128
args.z_dim = 5
args.hidden_dim = 100
args.ada_step_size = 0.05

args.num_stein_particles = 10
args.num_elbo_particles = 10




encoder_nn = encoder(args.hidden_dim, args.z_dim)
decoder_nn = decoder(args.hidden_dim, 28 * 28)
adam = optim.Adam(args.learning_rate)
ada = Adagrad(args.ada_step_size)


def non_stein(name):
    if name in ["encoder$params", "flow$params", "B"]:
        return True
    else:
        return False

method = SteinVI(
    model,
    guide,
    adam,
    RBFKernel(), 
    hidden_dim=args.hidden_dim, 
    z_dim=args.z_dim, 
    num_stein_particles=args.num_stein_particles, 
    num_elbo_particles=args.num_elbo_particles,
    loss_temperature=3,
    non_mixture_guide_params_fn= non_stein
    )



rng_key = PRNGKey(0)
train_init, train_fetch = load_dataset(
    MNIST, batch_size=args.batch_size, split="train"
)
test_init, test_fetch = load_dataset(
    MNIST, batch_size=args.batch_size, split="test"
)

num_train, train_idx = train_init()
rng_key, rng_key_binarize, rng_key_init = random.split(rng_key, 3)
sample_batch = binarize(rng_key_binarize, train_fetch(0, train_idx)[0])

state = method.init(rng_key_init, sample_batch)

In [8]:


@jit
def epoch_train(state, rng_key, train_idx):
    def body_fn(i, val):
        loss_sum, state = val
        rng_key_binarize = random.fold_in(rng_key, i)
        batch = binarize(rng_key_binarize, train_fetch(i, train_idx)[0])
        state, loss = method.update(state, batch)
        loss_sum += loss
        return loss_sum, state

    return lax.fori_loop(0, num_train, body_fn, (0.0, state))

@partial(jit, static_argnums=(4,))
def eval_data(state, rng_key, data_idx, num_test, data_fetch):
    def body_fun(i, loss_sum):
        rng_key_binarize = random.fold_in(rng_key, i)
        batch = binarize(rng_key_binarize, data_fetch(i, data_idx)[0])
        # FIXME: does this lead to a requirement for an rng_key arg in svi_eval?
        loss = method.evaluate(state, batch) / len(batch)
        loss_sum += loss
        return loss_sum

    loss = lax.fori_loop(0, num_test, body_fun, 0.0)
    loss = loss / num_test
    return loss



def encode_image(img, rng_key, particle = None):
    rng_key_binarize, rng_key_particle, rng_key_m, rng_key_z = jax.random.split(rng_key, 4)
    

    params = method.get_params(state)
    test_sample = binarize(rng_key_binarize, img)

    if particle == None:
        particle = jax.random.randint(rng_key_particle, 1, 0, args.num_stein_particles)
    a = jax.tree.map(lambda x: x[particle], params["a"])

    m = a

    encoder_params =  params["encoder$params"]
    

    z_mean, z_var =  encoder_nn[1](
        encoder_params, jnp.concat([test_sample.reshape([1, -1]), m.reshape([1, -1])], axis=1)
    )
    z = dist.Normal(z_mean, z_var).sample(rng_key_z)
    return z


def decode_image(z):
    params = method.get_params(state)
    decoder_params = params["decoder$params"]

    return decoder_nn[1](decoder_params, z).reshape([28, 28])


def reconstruct_img(epoch, rng_key, test_idx):
    img = test_fetch(0, test_idx)[0][0]
    plt.imsave(
        os.path.join(RESULTS_DIR, "original_epoch={}.png".format(epoch)),
        img,
        cmap="gray",
    )
    img_loc = decode_image(encode_image(img, rng_key))
    plt.imsave(
        os.path.join(RESULTS_DIR, "recons_epoch={}.png".format(epoch)),
        img_loc,
        cmap="gray",
    )
        
@jit
def mse_loss(targets: jnp.ndarray, preds: jnp.ndarray):
    def single_mse(target: jnp.ndarray, pred:jnp.ndarray):
        target = target.reshape(-1)
        pred = pred.reshape(-1)
        return jnp.mean((target - pred)**2)
    return jnp.mean(jax.vmap(single_mse)(targets, preds))


def epoch_conclude_mse(rng_key, test_idx):
    imgs = test_fetch(0, test_idx)[0]
    reconstructed = jax.vmap(lambda img, key: decode_image(encode_image(img, key)))(imgs, jax.random.split(rng_key, len(imgs)))
    return mse_loss(imgs, reconstructed)


def visualize_latent_space(rng_key, test_idx):
    imgs = test_fetch(0, test_idx)[0]
    ps = args.num_stein_particles
    l = len(imgs)

    colors = np.zeros((ps, l))
    zs = np.zeros((ps, l, args.z_dim))

    for particle in range(ps):
        rng_key, sub_key = jax.random.split(rng_key)
        colors[particle, :] = particle
        zs[particle] = np.array(jax.vmap(lambda img, key: encode_image(img, key, particle))(imgs, jax.random.split(sub_key, len(imgs)))).reshape((l, args.z_dim))
        
    colors = colors.flatten()
    zs = zs.reshape((ps*l, args.z_dim))
    # Now do TSNE

    zs_scaled = StandardScaler().fit_transform(zs)
    zs_pca = PCA(n_components=50).fit_transform(zs_scaled)
    
    
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca', learning_rate='auto')
    X_tsne = tsne.fit_transform(zs_pca)

    # 5. Visualize
    plt.figure(figsize=(10, 7))
    plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors, cmap='viridis', s=5)
    plt.colorbar()
    plt.title("t-SNE Visualization")
    plt.show()
        




In [9]:


for i in range(args.num_epochs):
    rng_key, rng_key_train, rng_key_test, rng_key_reconstruct = random.split(
        rng_key, 4
    )
    t_start = time.time()
    num_train, train_idx = train_init()
    
    _, state = epoch_train(state, rng_key_train, train_idx)
    rng_key, rng_key_test, rng_key_train, rng_key_reconstruct = random.split(rng_key, 4)
    num_test, test_idx = test_init()
    test_loss = eval_data(state, rng_key_test, test_idx, num_test, test_fetch)
    train_loss = eval_data(state, rng_key_train, train_idx, num_train, train_fetch)
    
    reconstruct_img(i, rng_key_reconstruct, test_idx)

    rng_key, rng_key_mse = jax.random.split(rng_key)
    mse_loss_val = epoch_conclude_mse(rng_key, test_idx)
    print(
        "Epoch {}: loss = {} ({:.2f} s.), mse_loss = {}".format(
            i, train_loss, time.time() - t_start, mse_loss_val
        )
    )
rng_key, rng_key_sample = random.split(rng_key)



Epoch 0: loss = 2032.989501953125 (67.05 s.), mse_loss = 0.060156356543302536
Epoch 1: loss = 1784.5687255859375 (17.13 s.), mse_loss = 0.057679541409015656
Epoch 2: loss = 2037.9686279296875 (16.84 s.), mse_loss = 0.05598588287830353
Epoch 3: loss = 4470.64599609375 (19.47 s.), mse_loss = 0.05162283405661583
Epoch 4: loss = nan (17.10 s.), mse_loss = nan


KeyboardInterrupt: 

C